# 01 · Exploring the Telugu speech data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pavanadityav-ai/telugu-asr-whisper/blob/main/notebooks/01_data_exploration.ipynb)

**Goal:** understand the data before training anything: how much audio there is, who is speaking, what the text looks like, and how Whisper's tokenizer handles Telugu.

**Runtime:** no GPU needed. *Runtime → Run all*. Takes about 5 minutes (mostly downloading 2.3 GB of audio).

In [ ]:
# Setup: install libraries, get the project code, connect Google Drive
!pip install -q -U datasets jiwer
import os, sys
if not os.path.isdir("telugu-asr-whisper"):
    !git clone -q https://github.com/pavanadityav-ai/telugu-asr-whisper
sys.path.insert(0, "telugu-asr-whisper")

from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/telugu-asr-whisper"  # results are saved here
os.makedirs(OUT, exist_ok=True)
print("Saving results to:", OUT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.audio_utils import load_fleurs_telugu, decode_audio

te = load_fleurs_telugu()
print(te)

## 1. How much audio do we have?
`num_samples` is the length of a clip in samples. The audio has 16,000 samples per second, so `seconds = num_samples / 16,000`. Whisper reads at most 30 seconds at a time, so we also count clips longer than that.

In [ ]:
rows = []
for split in ["train", "validation", "test"]:
    secs = np.array(list(te[split]["num_samples"])) / 16_000
    rows.append({"split": split, "clips": len(secs), "hours": secs.sum() / 3600,
                 "mean_sec": secs.mean(), "min_sec": secs.min(), "max_sec": secs.max(),
                 "over_30_sec": int((secs > 30).sum())})
summary = pd.DataFrame(rows).round(2)
summary

In [ ]:
train_secs = np.array(list(te["train"]["num_samples"])) / 16_000
plt.figure(figsize=(7, 3))
plt.hist(train_secs, bins=40)
plt.axvline(30, color="red", linestyle="--", label="Whisper's 30 s limit")
plt.xlabel("clip length (seconds)"); plt.ylabel("number of clips")
plt.title("Training clip lengths"); plt.legend(); plt.tight_layout()
plt.savefig(f"{OUT}/clip_lengths.png", dpi=150)
plt.show()

## 2. Who is speaking?

In [ ]:
gender = te["train"].features["gender"]
names = getattr(gender, "names", None)
values, counts = np.unique(list(te["train"]["gender"]), return_counts=True)
for v, c in zip(values, counts):
    print(f"{names[v] if names else v}: {c} clips ({100 * c / counts.sum():.0f}%)")

## 3. Listen to a clip

In [ ]:
from IPython.display import Audio, display

example = te["train"][0]
print(example["transcription"])
display(Audio(decode_audio(example["audio"]), rate=16_000))

## 4. What does the text look like?
Telugu glues many endings onto a word (నాయకత్వం → నాయకత్వంలో, "in the leadership"). So the same word appears in many forms, and many test words never appear in training. That is one reason we report **CER** (character error rate) next to WER.

In [ ]:
from src.text_utils import normalize_text, script_profile

train_text = [normalize_text(t) for t in te["train"]["transcription"]]
test_text = [normalize_text(t) for t in te["test"]["transcription"]]

words = np.array([len(t.split()) for t in train_text])
chars = np.array([len(t) for t in train_text])
print(f"Average sentence: {words.mean():.1f} words, {chars.mean():.0f} characters")

train_vocab = {w for t in train_text for w in t.split()}
test_words = [w for t in test_text for w in t.split()]
unseen = [w for w in test_words if w not in train_vocab]
print(f"Different words in training: {len(train_vocab):,}")
print(f"Test words never seen in training: {100 * len(unseen) / len(test_words):.0f}%")
print("Examples:", unseen[:10])

latin = sum(script_profile(t)["latin"] > 0 for t in train_text)
print(f"Training sentences containing English letters: {latin}")

# Save the training vocabulary for the error analysis notebook
with open(f"{OUT}/train_vocab.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(sorted(train_vocab)))

## 5. How does Whisper's tokenizer see Telugu?
Whisper turns text into **tokens** before learning it. Its tokenizer was built mostly from English, so Telugu needs many more tokens for the same meaning. Whisper also cannot write more than **448 tokens** for one clip.

In [ ]:
from transformers import WhisperTokenizer

tok = WhisperTokenizer.from_pretrained("openai/whisper-small")
english = "The first economic reforms happened under the leadership of Deng Xiaoping"
telugu = normalize_text(te["train"][0]["transcription"])
for name, s in [("English", english), ("Telugu", telugu)]:
    n = len(tok(s, add_special_tokens=False).input_ids)
    print(f"{name:8s} {len(s):3d} characters -> {n:3d} tokens  ({n / len(s):.2f} tokens per character)")

lens = np.array([len(tok(t).input_ids) for t in train_text])
print(f"\nTokens per Telugu training sentence: average {lens.mean():.0f}, longest {lens.max()}")
print("Sentences over the 448-token limit:", int((lens > 448).sum()))

## 6. Save a summary

In [ ]:
import json
data_summary = {
    "splits": summary.to_dict(orient="records"),
    "train_vocab_size": len(train_vocab),
    "test_unseen_word_rate": len(unseen) / len(test_words),
    "avg_tokens_per_sentence": float(lens.mean()),
}
with open(f"{OUT}/data_summary.json", "w") as f:
    json.dump(data_summary, f, indent=2)
data_summary

## My notes

_Write 3–5 things **you** noticed here, in your own words. These notes become the analysis section of the README and your SOP._

1. 
2. 
3. 